In [ ]:
import sys
import pathlib
cwd = pathlib.Path().resolve()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\srida\OneDrive\Desktop\repos\DisasterTweets-Classifier


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,)

from sklearn.metrics import accuracy_score, f1_score

from src.config import TWEETS_CSV


In [ ]:
df = pd.read_csv(TWEETS_CSV)

df = df[["text", "target"]].dropna()
df.head(), df["target"].value_counts()

(                                                text  target
 0  Communal violence in Bhainsa, Telangana. "Ston...       1
 1  Telangana: Section 144 has been imposed in Bha...       1
 2  Arsonist sets cars ablaze at dealership https:...       1
 3  Arsonist sets cars ablaze at dealership https:...       1
 4  "Lord Jesus, your love brings freedom and pard...       0,
 target
 0    9256
 1    2114
 Name: count, dtype: int64)

In [14]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"],
)

len(train_df), len(test_df)

(9096, 2274)

In [15]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("target", "labels")
test_ds = test_ds.rename_column("target", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 2274/2274 [00:00<00:00, 19481.65 examples/s]


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
)

training_args = TrainingArguments(
    output_dir="./distilbert_disaster",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
from transformers import Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import torch

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,   
    eval_dataset=test_ds,     
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


print("🚀 Training started...")
trainer.train()
print("🎉 Training completed!")

print("\n📊 Running evaluation...")
eval_results = trainer.evaluate()
print("\n📈 Evaluation Results:")
print(eval_results)

save_path = "./distilbert_disaster"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n💾 Model saved at: {save_path}")

def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = outputs.logits.softmax(dim=1)
        pred = torch.argmax(probs, dim=1).item()
    return {
        "label": int(pred),
        "probabilities": probs.numpy().tolist(),
    }

example_text = "The flood destroyed the entire village."
print("\n🔍 Example Prediction Input:")
print(example_text)

result = predict(example_text)

print("\n🧾 Prediction Output:")
print(result)


C:\Users\srida\AppData\Local\Temp\ipykernel_9812\363363778.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Training started...


c:\Users\srida\OneDrive\Desktop\repos\DisasterTweets-Classifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.429400
100,0.285500
150,0.275300
200,0.295900
250,0.261700
300,0.237400
350,0.256600
400,0.269700
450,0.241900
500,0.284800


c:\Users\srida\OneDrive\Desktop\repos\DisasterTweets-Classifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


🎉 Training completed!

📊 Running evaluation...


c:\Users\srida\OneDrive\Desktop\repos\DisasterTweets-Classifier\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



📈 Evaluation Results:
{'eval_loss': 0.24651998281478882, 'eval_accuracy': 0.9107299912049253, 'eval_f1': 0.7731843575418994, 'eval_runtime': 81.5426, 'eval_samples_per_second': 27.887, 'eval_steps_per_second': 0.883, 'epoch': 2.0}

💾 Model saved at: ./distilbert_disaster

🔍 Example Prediction Input:
The flood destroyed the entire village.

🧾 Prediction Output:
{'label': 1, 'probabilities': [[0.026290288195014, 0.973709762096405]]}
